In [1]:
import pandas as pd
import os


data_folder = os.path.join("../..", "data")
csv_name = "df_500k_processed.parquet"
csv_path = os.path.join(data_folder, csv_name)

df_500k = pd.read_parquet(csv_path)
df_500k.info()
display(df_500k.head())
df_500k.describe(include='all')
df_500k.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   DeviceId        500000 non-null  int64         
 1   StreetId        500000 non-null  int64         
 2   ArrivalTime     500000 non-null  datetime64[us]
 3   DepartureTime   500000 non-null  datetime64[us]
 4   Hour            500000 non-null  int32         
 5   WeekDay         500000 non-null  int32         
 6   Month           500000 non-null  int32         
 7   VehiclePresent  500000 non-null  bool          
dtypes: bool(1), datetime64[us](2), int32(3), int64(2)
memory usage: 21.5 MB


,DeviceId,StreetId,ArrivalTime,DepartureTime,Hour,WeekDay,Month,VehiclePresent
0,23915,123,2019-11-03 03:00:56,2019-11-03 03:07:26,3,6,11,False
1,23914,528,2019-04-16 14:14:47,2019-04-16 14:15:23,14,1,4,False
2,23913,670,2019-09-29 01:08:22,2019-09-29 01:31:41,1,6,9,False
3,23915,123,2019-06-05 18:59:17,2019-06-05 18:59:24,18,2,6,False
4,23915,123,2019-04-01 07:03:41,2019-04-01 07:14:12,7,0,4,True


DeviceId          0
StreetId          0
ArrivalTime       0
DepartureTime     0
Hour              0
WeekDay           0
Month             0
VehiclePresent    0
dtype: int64

## Remuestreo temporal para PyPots

In [2]:
# 1. Definimos un periodo de tiempo denso (ej. 3 meses: Enero, Febrero y Marzo)
fecha_inicio = '2019-01-01'
fecha_fin = '2019-03-31'

# 2. Filtramos el dataset original gigante quedándonos SOLO con ese bloque de tiempo intacto
mask_tiempo = (df_500k['ArrivalTime'] >= fecha_inicio) & (df_500k['DepartureTime'] <= fecha_fin)
df_500k_dense = df_500k[mask_tiempo].copy()
df_500k_dense.info()

# 3. (Opcional si aún es muy grande) Filtramos quedándonos con los 100 sensores más activos
sensores_top = df_500k_dense['DeviceId'].value_counts().head(100).index
print(f"Tamaño de la nueva muestra densa: {df_500k_dense.shape}")


<class 'pandas.DataFrame'>
Index: 102566 entries, 12 to 499999
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   DeviceId        102566 non-null  int64         
 1   StreetId        102566 non-null  int64         
 2   ArrivalTime     102566 non-null  datetime64[us]
 3   DepartureTime   102566 non-null  datetime64[us]
 4   Hour            102566 non-null  int32         
 5   WeekDay         102566 non-null  int32         
 6   Month           102566 non-null  int32         
 7   VehiclePresent  102566 non-null  bool          
dtypes: bool(1), datetime64[us](2), int32(3), int64(2)
memory usage: 5.2 MB
Tamaño de la nueva muestra densa: (102566, 8)


In [3]:
import numpy as np
import pandas as pd

# Trabajaremos con la muestra más densa
df_500k_mod = df_500k_dense.copy()

print("Verificando valores NA iniciales...")
print(df_500k_mod.isnull().sum())

print("Iniciando transformación a serie temporal continua...")


# 2. CREAR LA LÍNEA TEMPORAL MAESTRA (Ej. fotos cada 15 minutos)
tiempo_min = df_500k_mod['ArrivalTime'].min().floor('15min')
tiempo_max = df_500k_mod['DepartureTime'].max().ceil('15min')
grid_temporal = pd.date_range(start=tiempo_min, end=tiempo_max, freq='15min')

# 3. CREAR LA CUADRÍCULA (Sensor X Tiempo)
sensores_unicos = df_500k_mod['DeviceId'].unique()
df_grid = pd.MultiIndex.from_product(
    [sensores_unicos, grid_temporal],
    names=['DeviceId', 'Timestamp']
).to_frame(index=False)

# 4. PREPARAR DATAFRAMES PARA merge_asof
left_df = df_grid.sort_values(['Timestamp', 'DeviceId']).reset_index(drop=True)
right_df = (
    df_500k_mod[['DeviceId', 'ArrivalTime', 'DepartureTime', 'VehiclePresent', 'StreetId']]
    .sort_values(['ArrivalTime', 'DeviceId'])
    .reset_index(drop=True)
)

# 5. EL ESCÁNER (merge_asof)
df_500k_continuous = pd.merge_asof(
    left_df,
    right_df,
    left_on='Timestamp',
    right_on='ArrivalTime',
    by='DeviceId',
    direction='backward'
 )

# 6. CREACIÓN DE LOS HUECOS (NaN) PARA PYPOTS
condicion_vacio = df_500k_continuous['Timestamp'] > df_500k_continuous['DepartureTime']
df_500k_continuous['VehiclePresent'] = np.where(condicion_vacio, np.nan, df_500k_continuous['VehiclePresent'])

# 7. RECALCULAR VARIABLES TEMPORALES DESDE EL NUEVO GRID
df_500k_continuous['Hour'] = df_500k_continuous['Timestamp'].dt.hour
df_500k_continuous['WeekDay'] = df_500k_continuous['Timestamp'].dt.dayofweek.astype(int)
df_500k_continuous['Month'] = df_500k_continuous['Timestamp'].dt.month

# 8. LIMPIEZA FINAL
df_500k_continuous = df_500k_continuous.drop(columns=['ArrivalTime', 'DepartureTime'])
df_500k_continuous['StreetId'] = df_500k_continuous.groupby('DeviceId')['StreetId'].transform(lambda x: x.ffill().bfill())

print('Transformación completada sin errores.')
print(f'Tamaño de df_500k_continuous: {df_500k_continuous.shape}')
print("\nValores NA por columna en df_500k_continuous (¡El momento de la verdad!):")
print(df_500k_continuous.isnull().sum())

Verificando valores NA iniciales...
DeviceId          0
StreetId          0
ArrivalTime       0
DepartureTime     0
Hour              0
WeekDay           0
Month             0
VehiclePresent    0
dtype: int64
Iniciando transformación a serie temporal continua...
Transformación completada sin errores.
Tamaño de df_500k_continuous: (640875, 7)

Valores NA por columna en df_500k_continuous (¡El momento de la verdad!):
DeviceId               0
Timestamp              0
VehiclePresent    243454
StreetId               0
Hour                   0
WeekDay                0
Month                  0
dtype: int64


### One-Hot Encoding

In [5]:
print("Aplicando One-Hot Encoding a variables temporales...")

# 1. Definimos qué columnas categóricas vamos a "explotar"
categorical_cols = ['Hour', 'WeekDay', 'Month']

# 2. Convertimos a texto para que los nombres de las columnas queden limpios (ej. "Hour_15")
for col in categorical_cols:
    df_500k_continuous[col] = df_500k_continuous[col].astype(str)

# 3. La magia de Pandas (get_dummies)
df_final = pd.get_dummies(df_500k_continuous, columns=categorical_cols, dtype=int)

print("¡Transformación final completada!")
print(f"Nuevo tamaño del dataset: {df_final.shape}")
print("\nMuestra de las columnas resultantes:")
print(df_final.columns.tolist()[:10], "...") # Mostramos solo las primeras para no saturar
display(df_final.head())

Aplicando One-Hot Encoding a variables temporales...
¡Transformación final completada!
Nuevo tamaño del dataset: (640875, 38)

Muestra de las columnas resultantes:
['DeviceId', 'Timestamp', 'VehiclePresent', 'StreetId', 'Hour_0', 'Hour_1', 'Hour_10', 'Hour_11', 'Hour_12', 'Hour_13'] ...


,DeviceId,Timestamp,VehiclePresent,StreetId,Hour_0,Hour_1,Hour_10,Hour_11,Hour_12,Hour_13,...,WeekDay_0,WeekDay_1,WeekDay_2,WeekDay_3,WeekDay_4,WeekDay_5,WeekDay_6,Month_1,Month_2,Month_3
0,23914,2019-01-01,NaN,528.0,1,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,0
1,23915,2019-01-01,NaN,123.0,1,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,0
2,23916,2019-01-01,False,123.0,1,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,0
3,23917,2019-01-01,True,894.0,1,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,0
4,23918,2019-01-01,True,926.0,1,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,0
